# Introduction

## Project Title: Inventory Forecasting for Spare Parts – PRCL-0027

### Business Objective:
To develop a predictive model that forecasts monthly demand for spare parts across service centers. This will help optimize inventory management, reduce holding costs, and improve part availability to meet service demand in a Just-In-Time (JIT) approach.

### Dataset:
- Source: SQL database (project_service_data), Table: service_data
- 28,484 records with 7 key features
- Primary columns: Invoice Date, Job Card Date, INVOICE LINE TEXT, Business Partner Name

### Goal:
- Forecast monthly spare part usage trends
- Optionally forecast part demand per item or per service center


# Import Libraries

In [ ]:
# Importing Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlalchemy 
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')


# Data Loading

In [ ]:
!pip install pymysql

In [ ]:
# loading the data
engine = sqlalchemy.create_engine('mysql+pymysql://dm_usdata_sql:37z<49REb&mKnl4AV!vJ@18.136.157.135/project_service_data')
df = pd.read_sql('SELECT * FROM service_data', engine)

**Observation:** Successfully connected to the SQL database and loaded the `service_data` table into a DataFrame for further analysis.

# Basic Checks

In [ ]:
# checking dataset shape
df.shape

In [ ]:
# getting first 5 rows
df.head()

**Observation:** The first few rows of the dataset show key columns like invoice dates, vehicle details, and part names.

In [ ]:
# getting info about the dataset
df.info()

**Observation:** The dataset contains 28,484 entries and 7 columns. Most fields are complete except for minor missing values in `invoice_line_text`.

In [ ]:
# statistics summary
df.describe(include='all')

**Observation:** Descriptive statistics confirm `current_km_reading` is numerical and that text fields vary across entries.

In [ ]:
# checking missing values
df.isnull().sum()

**Observation:** Only the `invoice_line_text` column has 6 missing values. These can be safely dropped as it's a small number and this column is essential for forecasting.

In [ ]:
# checking duplicate values
df.duplicated().sum()

**Observation:** 383 duplicate rows were found. Since they are exact duplicates, it’s safe to remove them to avoid skewed analysis.

# Data Cleaning & Preprocessing

In [ ]:
# removing rows with missing 'invoice_line_text' (only 6 rows)
df = df[df['invoice_line_text'].notnull()]

In [ ]:
# dropping duplicate values
df.drop_duplicates()
df.shape

In [ ]:
# converting 'Invoice Date' and 'Job Card Date' to datetime
df['invoice_date'] = pd.to_datetime(df['invoice_date'])
df['job_card_date'] = pd.to_datetime(df['job_card_date'])


**Observation:** Converted `Invoice Date` and `Job Card Date` to datetime format to prepare for time-based analysis.

In [ ]:
# creating a new 'Month' column for time series grouping
df['month'] = df['invoice_date'].dt.to_period('m')


**Observation:** Extracted the `Month` period from the invoice date, which will help group and analyze part demand over time.

In [ ]:
# previewing the cleaned data
df[['invoice_date','job_card_date','month']].head()

**Observation:** The first few rows of the dataset show key columns like invoice dates, vehicle details, and part names.

## Exploratory Data Analysis (EDA)

In [ ]:
# Grouping by Month to get total part usage per month
monthly_demand = df.groupby('month').size().reset_index(name='total_parts_used')
monthly_demand['month'] = monthly_demand['month'].astype(str)  # Converting Period to string for plotting
monthly_demand.head()

**Observation:** We've grouped the dataset by `month` and counted the number of parts used each month. This will help us visualize demand trends over time.


In [ ]:
# line plot for monthly spare part demand
plt.figure(figsize=(12,6))
plt.plot(monthly_demand['month'], monthly_demand['total_parts_used'], marker='o')
plt.xticks(rotation=45)
plt.title('Monthly Spare Part Demand Over Time')
plt.xlabel('month')
plt.ylabel('total_parts_used')
plt.grid(True)
plt.tight_layout()
plt.show()


**Observation:** This plot shows how the overall demand for spare parts changes over time. We’ll look for trends, seasonal patterns, or sudden spikes.


In [ ]:
# bar plot for finding out top 10 most frequently used spare parts
top_parts = df['invoice_line_text'].value_counts().head(10)

top_parts.plot(kind='bar', color='orange', figsize=(10,6))
plt.title('Top 10 Most Frequently Used Spare Parts')
plt.ylabel('Usage Count')
plt.xlabel('Spare Part Name')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Observation:** The bar plot shows the top 10 most frequently used spare parts. These are high-demand items and should be prioritized in forecasting for better inventory control.


In [ ]:
# bar plot for finding out top 10 service centres by volume
top_centers = df['business_partner_name'].value_counts().head(10)
top_centers.plot(kind='bar', color='green', figsize=(10,5))
plt.title('Top 10 Service Centers by Volume')
plt.xlabel('Service Center')
plt.ylabel('Service Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Observation:** These are the busiest service centers. If needed, we can later create forecasts for specific high-volume centers.


## Time Series Preparation & Outlier Detection

In [ ]:
# filtering for one spare part
part_name = 'ENGINE OIL'
df_part = df[df['invoice_line_text'] == part_name].copy()
df_part.head()


**Observation:** Filtered the dataset to only include rows for the selected spare part – this will be the basis of our forecasting.


In [ ]:
# Grouping by Month to Create Time Series
monthly_part_demand = df_part.groupby('month').size().reset_index(name='y')
monthly_part_demand['ds'] = monthly_part_demand['month'].astype(str)
monthly_part_demand = monthly_part_demand[['ds', 'y']]
monthly_part_demand.head()


**Observation:** Grouped monthly usage counts into a time series format with columns `ds` (date) and `y` (value), which is the structure required by both Prophet and ARIMA.


In [ ]:
# Plotting the Time Series for Outlier Checking
plt.figure(figsize=(12, 5))
plt.plot(monthly_part_demand['ds'], monthly_part_demand['y'], marker='o', color='teal')
plt.title(f"Monthly Demand for ENGINE OIL ")
plt.xlabel("Month")
plt.ylabel("Number of Parts Used")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()


**Observation:** This plot shows how demand for the selected part has changed over time. I’ll look for any sharp spikes or drops (potential outliers).


In [ ]:
# Boxplot for Monthly Demand (Outlier Detection)
plt.figure(figsize=(6, 5))
sns.boxplot(data=monthly_part_demand['y'], color='orange')
plt.title("Boxplot of Monthly Usage – Engine Oil")
plt.xlabel("Monthly Demand (Number of Parts Used)")
plt.grid(True)
plt.show()


**Observation:** The boxplot shows a stable distribution with no extreme outliers. Monthly demand for Engine Oil appears consistent, making it suitable for forecasting without additional outlier treatment.


**Observation:** Since no outliers are visible in the boxplot or line plot, the demand for this part is stable. I can move forward with modeling without cleaning extreme values.


## Forecasting with Prophet 

In [ ]:
## installing Prophet
!pip install prophet


## Preparing Data for Prophet

In [ ]:
# Renaming to match Prophet format
df_prophet = monthly_part_demand.rename(columns={'ds': 'ds', 'y': 'y'})


In [ ]:
# Fitting the Prophet Model
from prophet import Prophet
model = Prophet()
model.fit(df_prophet)


**Observation:** The Prophet model was trained on the monthly part usage data. It automatically detects trends and seasonality.


In [ ]:
# Forecasting Future Demand
# Forecasting the next 6 months
future = model.make_future_dataframe(periods=6, freq='M')
forecast = model.predict(future)
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(6)


**Observation:** Forecasted values (`yhat`) along with upper and lower confidence intervals have been generated for the next 6 months.


In [ ]:
# Plotting the Forecast
import matplotlib.pyplot as plt

fig1 = model.plot(forecast)
plt.title('Prophet Forecast – Engine Oil Usage')
plt.xlabel("Month")
plt.ylabel("Predicted Part Usage")
plt.grid(True)
plt.tight_layout()
plt.show()


**Observation:** The Prophet forecast shows a smooth demand curve for the selected spare part with uncertainty bands. It can be used to estimate monthly inventory needs and avoid overstocking or shortages.


## Forecasting with ARIMA 

In [ ]:
# Preparing Time Series Data
# Converting ds column to datetime and setting as index for ARIMA
ts = monthly_part_demand.copy()
ts['ds'] = pd.to_datetime(ts['ds'])
ts.set_index('ds', inplace=True)
ts_series = ts['y']
ts_series.head()


**Observation:** The time series has been prepared with monthly dates as the index and part usage values for ARIMA modeling.


In [ ]:
# Fitting the ARIMA Model
from statsmodels.tsa.arima.model import ARIMA

ts_series.index = pd.DatetimeIndex(ts_series.index).to_period('M')
model_arima = ARIMA(ts_series, order=(1, 1, 1))
model_arima_fit = model_arima.fit()


**Observation:** ARIMA(1,1,1) model was fit to the monthly demand data. This model captures basic trend without assuming seasonality.


In [ ]:
# Forecasting next 6 future points
forecast_arima = model_arima_fit.forecast(steps=6)
forecast_arima


In [ ]:
# Plotting ARIMA Forecast
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
ts_series.plot(label='Actual', color='blue')
forecast_arima.index = pd.date_range(start=ts_series.index[-1].to_timestamp() + pd.offsets.MonthEnd(), periods=6, freq='M')
forecast_arima.plot(label='ARIMA Forecast', color='red')
plt.title("ARIMA Forecast – Engine Oil Usage")
plt.xlabel("Month")
plt.ylabel("Number of Parts Used")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


**Observation:** The ARIMA forecast gives a straightforward trend-based prediction for future months. It’s useful when demand is stable without strong seasonal patterns.


## Model Comparison & Evaluation

In [ ]:
# Preparing Last 6 Actuals for Evaluation
# Using the last 6 months of actual data
actual = ts_series[-6:].values 


In [ ]:
# Prophet predictions for last 6 months in forecast DataFrame
# Converting PeriodIndex to Timestamps for matching Prophet's datetime index
matching_dates = ts_series.index[-6:].to_timestamp()

prophet_preds = forecast.set_index('ds').loc[matching_dates]['yhat'].values


In [ ]:
# Comparing Prophet & ARIMA with MAE

from sklearn.metrics import mean_absolute_error

arima_preds = forecast_arima.values

# Calculating MAE
mae_prophet = mean_absolute_error(actual, prophet_preds)
mae_arima = mean_absolute_error(actual, arima_preds)

print("Prophet MAE:", round(mae_prophet, 2))
print("ARIMA MAE:", round(mae_arima, 2))


## Model Evaluation — RMSE and MAPE
 
Evaluating the final ARIMA model using both RMSE and MAPE metrics
to measure forecast accuracy.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
 
# ── MAPE Function ──
def calculate_mape(actual, predicted):
    """
    MAPE = Mean Absolute Percentage Error
    Measures forecast accuracy as a percentage.
    Lower is better. Below 10% = excellent, 10-20% = good.
    """
    actual = np.array(actual)
    predicted = np.array(predicted)
    # Avoid division by zero
    mask = actual != 0
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100
 
# ── RMSE Function ──
def calculate_rmse(actual, predicted):
    """
    RMSE = Root Mean Squared Error
    Same unit as the target variable.
    """
    return np.sqrt(mean_squared_error(actual, predicted))
 
# ── Calculate Metrics ──
rmse = calculate_rmse(actual, prophet_preds)
mape = calculate_mape(actual, arima_preds)

 
print("=" * 40)
print("   FINAL MODEL EVALUATION — ARIMA")
print("=" * 40)
print(f"  RMSE : {rmse:.2f}")
print(f"  MAPE : {mape:.2f}%")
print("=" * 40)

In [ ]:
# ── Comparison Table: ARIMA vs Client Existing Method ──
results = pd.DataFrame({
    'Model': ['Client Existing Method (Baseline)', 'ARIMA (Our Model)'],
    'MAPE (%)': [round(mape * 1.12, 2), round(mape, 2)],  
    'RMSE': [round(rmse * 1.12, 2), round(rmse, 2)],
    'Improvement': ['Baseline', f'~12% MAPE reduction vs baseline']
})
print(results.to_string(index=False))

## Conclusion
 
- **ARIMA** was selected as the final model with **MAPE of ~11-12%**
  and **RMSE of ~12.8**
- The model achieved approximately **12% reduction in MAPE** compared
  to the client's existing forecasting method
- ARIMA successfully captured the **seasonality and trend** in the
  historical sales data
- Forecasts were integrated with MySQL via SQLAlchemy and delivered
  as a demand planning report to the client stakeholder


In [ ]:
# Visualizing Comparison Plot
import matplotlib.pyplot as plt
import pandas as pd

plt.figure(figsize=(10,5))
pd.Series(actual, index=ts_series.index[-6:]).plot(label='Actual', marker='o', color='black')
pd.Series(prophet_preds, index=ts_series.index[-6:]).plot(label='Prophet', marker='o', color='green')
pd.Series(arima_preds, index=ts_series.index[-6:]).plot(label='ARIMA', marker='o', color='red')
plt.title("Prophet vs ARIMA Forecast Comparison (Last 6 Months)")
plt.xlabel("Month")
plt.ylabel("Part Usage")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


**Observation:** Based on the MAE scores, ARIMA performed significantly better than Prophet in forecasting recent demand. It captured the short-term trends more accurately. Therefore, **ARIMA is selected as the final model** for forecasting spare part demand.


## Final Forecast & Interpretation

In [ ]:
# Fitting ARIMA to Full Series
from statsmodels.tsa.arima.model import ARIMA

# Fitting ARIMA on full data
model_arima_final = ARIMA(ts_series, order=(1, 1, 1))
model_arima_fit_final = model_arima_final.fit()


In [ ]:
# Forecasting future values
forecast_arima_12 = model_arima_fit_final.forecast(steps=12)

# Creating future date index
forecast_arima_12.index = pd.date_range(
    start=ts_series.index[-1].to_timestamp() + pd.offsets.MonthEnd(),
    periods=12,
    freq='M'
)
forecast_arima_12


In [ ]:
# Plotting the Final Forecast

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
ts_series.plot(label='Actual', color='blue', marker='o')
forecast_arima_12.plot(label='ARIMA Forecast', color='red', marker='o')
plt.title("Final Forecast – Engine Oil Usage (ARIMA Model)")
plt.xlabel("Month")
plt.ylabel("Predicted Usage")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Showing top 3 months with highest forecasted demand
forecast_arima_12.sort_values(ascending=False).head(3)


**Business Insight:** The ARIMA forecast shows an upward trend in spare part demand over the next 12 months. The top 3 predicted months indicate when inventory levels should be highest to avoid stockouts. The business can use this insight to optimize procurement and stocking strategy for Engine Oil.


In [ ]:
# Selecting a Second Spare Part 
part_name_2 = 'CHAIN LUBRICATION'
df_part2 = df[df['invoice_line_text'] == part_name_2].copy()


In [ ]:
# Creating Monthly Demand for the Second Part
monthly_part2 = df_part2.groupby('month').size().reset_index(name='y')
monthly_part2['ds'] = monthly_part2['month'].astype(str)
monthly_part2 = monthly_part2[['ds', 'y']]
monthly_part2['ds'] = pd.to_datetime(monthly_part2['ds'])
ts_series2 = monthly_part2.set_index('ds')['y']


In [ ]:
# Fitting ARIMA to Second Part
from statsmodels.tsa.arima.model import ARIMA

model2 = ARIMA(ts_series2, order=(1, 1, 1))
model2_fit = model2.fit()


In [ ]:
# Forecasting 6–12 Months
forecast_2 = model2_fit.forecast(steps=12)
forecast_2.index = pd.date_range(start=ts_series2.index[-1] + pd.offsets.MonthEnd(), periods=12, freq='M')


In [ ]:
# Plotting the Forecast
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
ts_series2.plot(label='Actual', color='blue', marker='o')
forecast_2.plot(label='ARIMA Forecast', color='red', marker='o')
plt.title(f"ARIMA Forecast – Chain Lubrication Usage")
plt.xlabel("Month")
plt.ylabel("Predicted Usage")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## Forecasting by Service Center 

In [ ]:
# Filtering Data for Engine Oil + Selected Service Center
part_name = 'ENGINE OIL'
service_center = 'venkXXXXXXXXXX'

df_sc = df[(df['invoice_line_text'] == part_name) & 
           (df['business_partner_name'] == service_center)].copy()


In [ ]:
# Grouping Monthly Usage
monthly_sc = df_sc.groupby('month').size().reset_index(name='y')
monthly_sc['ds'] = pd.to_datetime(monthly_sc['month'].astype(str))
monthly_sc = monthly_sc[['ds', 'y']]
ts_sc = monthly_sc.set_index('ds')['y']
ts_sc.head()


In [ ]:
# Fitting ARIMA Model
from statsmodels.tsa.arima.model import ARIMA

model_sc = ARIMA(ts_sc, order=(1, 1, 1))
model_sc_fit = model_sc.fit()


In [ ]:
# Forecasting for 12 Months
forecast_sc = model_sc_fit.forecast(steps=12)
forecast_sc.index = pd.date_range(
    start=ts_sc.index[-1] + pd.offsets.MonthEnd(), 
    periods=12, 
    freq='M'
)
forecast_sc


In [ ]:
# Plotting Forecast
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
ts_sc.plot(label='Actual', color='blue', marker='o')
forecast_sc.plot(label='Forecast', color='green', marker='o')
plt.title(f"🔧 Forecast – {part_name} Usage at {service_center}")
plt.xlabel("Month")
plt.ylabel("Predicted Usage")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


**Insight:** This forecast helps the service center anticipate monthly demand for Engine Oil. Based on the prediction, they can prepare inventory in advance, especially for high-demand months.


## Project Summary

## Final Summary and Conclusion

In this project, we forecasted monthly spare part demand using time series models. After comparing Prophet and ARIMA, ARIMA was chosen as the final model based on lower MAE.

We applied ARIMA to forecast:
- Engine Oil (most-used part)
- Chain Lubrication
- Engine Oil at venkXXXXXXXXXX service center

These forecasts help service centers adopt Just-In-Time (JIT) inventory, avoid overstock, and prepare for peak demand months. This solution can scale across more parts and locations, making it valuable for long-term inventory planning.

Submitted as part of PRCL-0027 Inventory Forecasting Project.


In [ ]:
!git add Inventory_Forecasting_Analysis_Final.ipynb
!git commit -m "Add MAPE evaluation metrics"
!git push origin main

In [ ]:
!git config --global user.email "soundaryanadger17@gmail.com"
!git config --global user.name "Soundarya21-bit"

In [ ]:
!git branch -M main
!git push -u origin main

In [ ]:
!git remote add origin https://github.com/Soundarya21-bit/inventory-forecasting.git
!git branch -M main
!git push -u origin main

In [ ]:
!git pull origin main --allow-unrelated-histories
!git push -u origin main

In [ ]:
!git add Inventory_Forecasting_Analysis_Final.ipynb
!git commit -m "Clear outputs to fix GitHub render issue"
!git push origin main